# Task 1: Binary Classification using KNN from Scratch
This notebook follows the assignment and builds KNN without sklearn.

## 1. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
plt.rcParams['figure.figsize']=(8,5)

## 2. Load Dataset

In [ ]:
df=pd.read_csv('/content/data.csv')  # In Colab upload data.csv first
df.head()

## 3. Clean Data

In [ ]:
df=df.drop(columns=['id','Unnamed: 32'],errors='ignore')
df['diagnosis']=df['diagnosis'].map({'M':1,'B':0})
X=df.drop('diagnosis',axis=1).values
y=df['diagnosis'].values
print(X.shape,y.shape)

## 4. Train/Test Split (80/20)

In [ ]:
np.random.seed(42)
idx=np.random.permutation(len(X))
split=int(0.8*len(X))
train_idx,test_idx=idx[:split],idx[split:]
X_train,X_test=X[train_idx],X[test_idx]
y_train,y_test=y[train_idx],y[test_idx]
print(len(X_train),len(X_test))

## 5. Standardization

In [ ]:
mean=X_train.mean(axis=0)
std=X_train.std(axis=0)
X_train=(X_train-mean)/std
X_test=(X_test-mean)/std

## 6. Distance Functions

In [ ]:
def euclidean(a,b):
    return np.sqrt(np.sum((a-b)**2))

def manhattan(a,b):
    return np.sum(np.abs(a-b))

def minkowski(a,b,p=3):
    return np.sum(np.abs(a-b)**p)**(1/p)

def cosine_distance(a,b):
    num=np.dot(a,b)
    den=np.linalg.norm(a)*np.linalg.norm(b)
    if den==0: return 1
    return 1-num/den

def hamming_distance(a,b):
    return np.mean(np.round(a,3)!=np.round(b,3))

## 7. KNN Classifier

In [ ]:
def knn_predict(Xtr,ytr,x,k,metric):
    d=[]
    for i in range(len(Xtr)):
        if metric=='euclidean':
            dist=euclidean(x,Xtr[i])
        elif metric=='manhattan':
            dist=manhattan(x,Xtr[i])
        elif metric=='minkowski':
            dist=minkowski(x,Xtr[i])
        elif metric=='cosine':
            dist=cosine_distance(x,Xtr[i])
        else:
            dist=hamming_distance(x,Xtr[i])
        d.append((dist,ytr[i]))
    d=sorted(d,key=lambda t:t[0])[:k]
    return Counter([c for _,c in d]).most_common(1)[0][0]

def predict_dataset(Xtr,ytr,Xte,k,metric):
    return np.array([knn_predict(Xtr,ytr,x,k,metric) for x in Xte])

def accuracy(y,yhat):
    return np.mean(y==yhat)

## 8. Experiment

In [ ]:
Ks=[3,4,9,20,47]
metrics=['euclidean','manhattan','minkowski','cosine','hamming']
results={}
for m in metrics:
    results[m]=[]
    for k in Ks:
        pred=predict_dataset(X_train,y_train,X_test,k,m)
        acc=accuracy(y_test,pred)
        results[m].append(acc)
        print(m,k,round(acc,4))

## 9. Plot

In [ ]:
for m in metrics:
    plt.plot(Ks,results[m],marker='o',label=m)
plt.xlabel('K')
plt.ylabel('Accuracy')
plt.title('K vs Accuracy')
plt.legend()
plt.grid(True)
plt.show()

## 10. Best Model

In [ ]:
best_metric,best_k,best_acc=None,None,-1
for m in metrics:
    for i,k in enumerate(Ks):
        if results[m][i]>best_acc:
            best_acc=results[m][i]
            best_metric=m
            best_k=k
print(best_metric,best_k,best_acc)
y_pred=predict_dataset(X_train,y_train,X_test,best_k,best_metric)

## 11. Confusion Matrix, Precision, Recall

In [ ]:
def confusion_matrix(y,yhat):
    tp=np.sum((y==1)&(yhat==1))
    tn=np.sum((y==0)&(yhat==0))
    fp=np.sum((y==0)&(yhat==1))
    fn=np.sum((y==1)&(yhat==0))
    return np.array([[tn,fp],[fn,tp]])

cm=confusion_matrix(y_test,y_pred)
tn,fp=cm[0]
fn,tp=cm[1]
precision=tp/(tp+fp)
recall=tp/(tp+fn)
print(cm)
print('Precision',round(precision,4))
print('Recall',round(recall,4))
print('Accuracy',round(best_acc,4))

## 12. Bonus Decision Boundary

In [ ]:
f1,f2=0,1
Xt=X_train[:,[f1,f2]]
Xe=X_test[:,[f1,f2]]

def predict2(Xtr,ytr,Xte,k,m):
    return np.array([knn_predict(Xtr,ytr,x,k,m) for x in Xte])

x_min,x_max=Xt[:,0].min()-1,Xt[:,0].max()+1
y_min,y_max=Xt[:,1].min()-1,Xt[:,1].max()+1
xx,yy=np.meshgrid(np.linspace(x_min,x_max,200),np.linspace(y_min,y_max,200))
grid=np.c_[xx.ravel(),yy.ravel()]
Z=predict2(Xt,y_train,grid,best_k,best_metric).reshape(xx.shape)
plt.contourf(xx,yy,Z,alpha=0.3)
plt.scatter(Xt[:,0],Xt[:,1],c=y_train,s=18)
plt.xlabel(df.columns[1]); plt.ylabel(df.columns[2]); plt.title('Decision Boundary')
plt.show()